# E/I Chain Signature EDA

## tl;dr

This notebook performs exploratory analysis of directed simple 3-node chains in `mij_netlist.csv`. It summarizes all eight E/I chain signatures (`EEE`, `EEI`, `EIE`, `EII`, `IEE`, `IEI`, `IIE`, `III`) by chain count, count share, absolute chain weight, weight share, signed chain weight, and geometric-mean intensity. It also breaks chains down by anatomical region, cell type with region prefixes removed, region/cell-type pathways, and two-edge region flow.

Run the notebook top to bottom to refresh all CSV tables and PNG visualizations in `outputs/ei_chain_signature_eda/`.

## Context & Methods

### Key Assumptions

- The input edge list uses `pre_neuron`, `post_neuron`, `pre_ei`, `post_ei`, and `m_ij`.
- A chain is a directed simple 3-node path `source -> middle -> target`; repeated nodes such as `A -> B -> A` are excluded.
- The E/I signature is based on the source, middle, and target neuron labels.
- Absolute chain weight is `abs(m_ij_1 * m_ij_2)`.
- Signed chain weight is `m_ij_1 * m_ij_2`.
- Chain intensity is `sqrt(abs(m_ij_1 * m_ij_2))`, following weighted-motif intensity intuition.
- Region is parsed as the first token of the neuron label, such as `DG`, `CA3`, or `CA1`.
- Cell type is parsed as the neuron label after removing that first region token.

In [ ]:
from pathlib import Path
from IPython.display import display

from ei_chain_signature import ChainSignatureConfig, run_chain_signature_analysis

config = ChainSignatureConfig(
    data_path=Path("mij_netlist.csv"),
    output_dir=Path("outputs") / "ei_chain_signature_eda",
    selected_signature="EII",
    top_n=20,
)

print(f"Reading: {Path(config.data_path).resolve()}")
print(f"Writing tables to: {config.table_dir.resolve()}")
print(f"Writing figures to: {config.figure_dir.resolve()}")

## Data

In [ ]:
result = run_chain_signature_analysis(config, make_figures=True, show_figures=True)

display(result.tables["data_profile"])
display(result.tables["chain_profile"])

## Results

In [ ]:
display(result.tables["signature_summary"])

In [ ]:
display(result.tables["region_makeup_by_signature"].head(20))
display(result.tables["region_role_makeup_by_signature"].head(20))

In [ ]:
display(
    result.tables["cell_type_makeup_by_signature"]
    .sort_values("abs_chain_weight", ascending=False)
    .head(20)
)
display(result.tables["cell_type_role_makeup_by_signature"].head(20))

In [ ]:
display(result.tables["region_pathways_by_signature"].head(20))
display(result.tables["cell_type_pathways_by_signature"].head(20))

In [ ]:
display(result.tables["region_to_region_two_edge_flow"].head(20))
display(result.tables[f"high_weight_{config.selected_signature}_chains"].head(20))

In [ ]:
print("Tables written:")
for name, path in result.table_paths.items():
    print(f"- {name}: {path}")

print("\nFigures written:")
for name, path in result.figure_paths.items():
    print(f"- {name}: {path}")

## Takeaways

Use `signature_summary.csv` to compare signature prevalence versus chain-weight concentration. The `weight_vs_count_pct_point_gap` column highlights signatures whose chain-weight share is disproportionate relative to their count share.

Use the region and cell-type heatmaps to distinguish broad anatomical or cell-class composition from the role-specific source, middle, and target breakdowns. The pathway tables preserve exact high-weight chains for audit and drill-down.